# Spectral Bipartition for Community Detection in the Karate Club Network

This notebook implements the **Spectral Bipartition method** to detect communities in Zachary's famous Karate Club network. The method uses eigenvalue decomposition of the modularity matrix to optimally partition the network into two communities.

## Section 1: Import Libraries and Load Data

Import all required libraries for network analysis, linear algebra, and visualization.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy import sparse

print("✓ Libraries imported successfully")

: 

Load Zachary's Karate Club network - a classic dataset containing 34 members and 78 friendship connections. The network split into two communities (led by "Mr. Hi" and "Admin") during an actual club dispute.

In [ ]:
# Load the network
G = nx.karate_club_graph()
n = G.number_of_nodes()
m = G.number_of_edges()

print(f"Network Statistics:")
print(f"  • Nodes: {n}")
print(f"  • Edges: {m}")
print(f"  • Average Degree: {2*m/n:.2f}")

# Visualize the original network
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=300)
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Karate Club Network - Original", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Section 2: Compute Matrices for Modularity Analysis

**Adjacency Matrix (A)**: Encodes actual edges. $A_{ij} = 1$ if nodes $i$ and $j$ are connected, else 0.

**Probability Matrix (P)**: Expected edges under the null model. $P_{ij} = \frac{k_i k_j}{2m}$ where $k_i$ is the degree of node $i$ and $m$ is total edges.

**Modularity Matrix (B)**: Deviation from null model. $B = A - P$ captures the "surprising" structure in the network.

In [ ]:
# Compute adjacency matrix as numpy array
A = np.array(nx.adjacency_matrix(G).todense(), dtype=np.float64)

# Get degree sequence efficiently
degrees = np.array(list(dict(G.degree()).values()), dtype=np.float64)

# Compute probability matrix
P = np.outer(degrees, degrees) / (2 * m)

# Compute modularity matrix
B = A - P

print("Matrix Dimensions:")
print(f"  • Adjacency Matrix (A): {A.shape}")
print(f"  • Probability Matrix (P): {P.shape}")
print(f"  • Modularity Matrix (B): {B.shape}")
print(f"\nMatrix Properties:")
print(f"  • B is symmetric: {np.allclose(B, B.T)}")
print(f"  • B trace (should be ~0): {np.trace(B):.6f}")

## Section 3: Spectral Decomposition

The optimal partition maximizes $Q = \frac{1}{4m} s^T B s$ where $s \in \{-1, +1\}^n$.

By the **Rayleigh-Ritz theorem**, this is solved by the **leading eigenvector** of $B$. We use `scipy.linalg.eigh()` for improved numerical stability on symmetric matrices.

In [ ]:
# Improved eigendecomposition for symmetric matrix
from scipy import linalg

# eigh() is more stable for symmetric matrices than eig()
eigenvalues, eigenvectors = linalg.eigh(B)

# Get the leading eigenvalue and eigenvector (largest eigenvalue is last)
lambda_1 = eigenvalues[-1]
u_1 = eigenvectors[:, -1].real  # Extract real part

print("Spectral Analysis Results:")
print(f"  • Leading Eigenvalue (λ₁): {lambda_1:.6f}")
print(f"  • Eigenvector range: [{u_1.min():.4f}, {u_1.max():.4f}]")
print(f"  • Eigenvector L2-norm: {np.linalg.norm(u_1):.6f}")

# Show top 5 eigenvalues
print(f"\nTop 5 Eigenvalues:")
for i, ev in enumerate(eigenvalues[-5:][::-1], 1):
    print(f"  {i}. λ = {ev:.6f}")

## Section 4: Community Assignment via Thresholding

Convert the continuous eigenvector to discrete community labels using simple thresholding:
- Nodes with $u_1(i) > 0$ → **Community A**
- Nodes with $u_1(i) \leq 0$ → **Community B**

In [ ]:
# Threshold eigenvector to get binary community assignment
s = np.where(u_1 > 0, 1, -1)

# Count community sizes
size_A = np.sum(s == 1)
size_B = np.sum(s == -1)

print("Community Assignment:")
print(f"  • Community A (s=+1): {size_A} nodes")
print(f"  • Community B (s=-1): {size_B} nodes")
print(f"  • Partition: {size_A}/{size_B} split")

## Section 5: Compute and Validate Modularity Score

The modularity score $Q = \frac{1}{4m} s^T B s$ quantifies how well the partition captures community structure:
- **Q > 0**: Partition is better than random chance
- **Q ∈ [0.3, 0.7]**: Strong community structure detected
- **Q > 0.7**: Very strong community structure

In [ ]:
# Calculate modularity score
Q = (1 / (4 * m)) * (s @ B @ s)

print("Modularity Analysis:")
print(f"  • Modularity Score (Q): {Q:.6f}")
print(f"  • Leading Eigenvalue (λ₁): {lambda_1:.6f}")
print(f"  • Expected Q from λ₁: {lambda_1 / (4*m):.6f}")

# Validate result
if Q > 0:
    print(f"\n✓ VALID: Positive modularity indicates meaningful community structure!")
    if Q > 0.3:
        print(f"✓ STRONG: Community structure is statistically significant (Q > 0.3)")
    if Q > 0.7:
        print(f"✓ EXCELLENT: Very strong community structure detected (Q > 0.7)")
else:
    print(f"\n✗ WARNING: Negative modularity - partition worse than random!")

# Visualize communities
plt.figure(figsize=(14, 6))

# Left: Community partition
ax1 = plt.subplot(1, 2, 1)
colors = ['#FF6B6B' if s[i] == 1 else '#4ECDC4' for i in range(n)]
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=300, ax=ax1)
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.3, ax=ax1)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax1)
ax1.set_title(f"Spectral Bipartition\n(Q = {Q:.4f})", fontsize=12, fontweight='bold')
ax1.axis('off')

# Right: Eigenvalue spectrum
ax2 = plt.subplot(1, 2, 2)
ax2.plot(eigenvalues, 'o-', markersize=4, linewidth=1, color='steelblue')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Zero')
ax2.scatter([n-1], [lambda_1], color='red', s=100, zorder=5, label=f'λ₁ = {lambda_1:.4f}')
ax2.set_xlabel('Eigenvalue Index', fontsize=11)
ax2.set_ylabel('Eigenvalue', fontsize=11)
ax2.set_title('Eigenvalue Spectrum of B', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

# Extract node lists for each community
nodes_list = list(G.nodes())
comm_A = sorted([nodes_list[i] for i in range(n) if s[i] == 1])
comm_B = sorted([nodes_list[i] for i in range(n) if s[i] == -1])

print(f"\nCommunity A (Red) - {len(comm_A)} members:")
print(f"  {comm_A}")
print(f"\nCommunity B (Teal) - {len(comm_B)} members:")
print(f"  {comm_B}")

## Section 6: Recursive Bisection for Hierarchical Community Detection

The **Recursive Bisection** method repeatedly applies spectral bipartition:
1. Start with the whole graph
2. If modularity Q > 0, split into two communities
3. Recursively apply the method to each new community
4. Stop when Q ≤ 0 (no improvement from further splitting)

This reveals the **hierarchical structure** of the network.

In [ ]:
# Define recursive bisection function
def spectral_bisection_recursive(nodes_list, graph, pos=None, depth=0, parent_id=""):
    """
    Recursively apply spectral bipartition to detect hierarchical communities.
    
    Parameters:
    -----------
    nodes_list : list
        Nodes in the current subgraph
    graph : networkx.Graph
        The full network graph
    pos : dict
        Precomputed layout positions for visualization
    depth : int
        Current recursion depth (for visualization)
    parent_id : str
        Identifier of parent community
    
    Returns:
    --------
    dict : Hierarchical community structure
    """
    
    # Get subgraph
    subgraph = graph.subgraph(nodes_list).copy()
    n_sub = subgraph.number_of_nodes()
    m_sub = subgraph.number_of_edges()
    
    # Base case: too small to split
    if n_sub <= 2:
        return {"nodes": nodes_list, "children": [], "modularity": 0, "depth": depth}
    
    # Compute matrices for subgraph
    node_mapping = {i: node for i, node in enumerate(nodes_list)}
    A_sub = np.array(nx.adjacency_matrix(subgraph).todense(), dtype=np.float64)
    degrees_sub = np.array(list(dict(subgraph.degree()).values()), dtype=np.float64)
    
    # Compute modularity matrix
    P_sub = np.outer(degrees_sub, degrees_sub) / (2 * m_sub) if m_sub > 0 else np.zeros_like(A_sub)
    B_sub = A_sub - P_sub
    
    # Eigendecomposition
    eigenvalues_sub, eigenvectors_sub = linalg.eigh(B_sub)
    lambda_max = eigenvalues_sub[-1]
    u_max = eigenvectors_sub[:, -1].real
    
    # Community assignment and modularity
    s_sub = np.where(u_max > 0, 1, -1)
    Q_sub = (1 / (4 * m_sub)) * (s_sub @ B_sub @ s_sub) if m_sub > 0 else 0
    
    # Visualize current state
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Get community colors
    colors_sub = ['#FF6B6B' if s_sub[i] == 1 else '#4ECDC4' for i in range(n_sub)]
    
    # Use precomputed positions or compute new ones
    if pos is None:
        pos_sub = nx.spring_layout(subgraph, seed=42, k=0.5)
    else:
        pos_sub = {node: pos[node] for node in nodes_list if node in pos}
    
    # Draw
    nx.draw_networkx_nodes(subgraph, pos_sub, node_color=colors_sub, node_size=300, ax=ax)
    nx.draw_networkx_edges(subgraph, pos_sub, edge_color='gray', alpha=0.3, ax=ax)
    nx.draw_networkx_labels(subgraph, pos_sub, font_size=8, ax=ax)
    
    # Title
    comm_a_size = np.sum(s_sub == 1)
    comm_b_size = np.sum(s_sub == -1)
    ax.set_title(f"Iteration {depth} | Nodes: {n_sub} | Partition: {comm_a_size}/{comm_b_size} | Q = {Q_sub:.4f}", 
                 fontsize=12, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*70}")
    print(f"Recursion Level {depth} (Community {parent_id})")
    print(f"{'='*70}")
    print(f"  • Nodes: {n_sub} | Edges: {m_sub}")
    print(f"  • Modularity (Q): {Q_sub:.6f}")
    print(f"  • Leading Eigenvalue: {lambda_max:.6f}")
    
    # Decide: continue recursion if Q > 0
    if Q_sub > 0:
        print(f"  ✓ Positive modularity → Split into 2 communities")
        
        # Split nodes
        nodes_a = sorted([nodes_list[i] for i in range(n_sub) if s_sub[i] == 1])
        nodes_b = sorted([nodes_list[i] for i in range(n_sub) if s_sub[i] == -1])
        
        print(f"  • Community A: {len(nodes_a)} nodes → {nodes_a}")
        print(f"  • Community B: {len(nodes_b)} nodes → {nodes_b}")
        
        # Recursive calls
        child_a = spectral_bisection_recursive(nodes_a, graph, pos, depth+1, parent_id + "A")
        child_b = spectral_bisection_recursive(nodes_b, graph, pos, depth+1, parent_id + "B")
        
        return {
            "nodes": nodes_list,
            "children": [child_a, child_b],
            "modularity": Q_sub,
            "depth": depth,
            "size": n_sub
        }
    else:
        print(f"  ✗ Non-positive modularity → Stop splitting (final community)")
        print(f"  • Final Community: {nodes_list}")
        
        return {
            "nodes": nodes_list,
            "children": [],
            "modularity": Q_sub,
            "depth": depth,
            "size": n_sub,
            "final": True
        }

# Run recursive bisection
print("\n" + "="*70)
print("HIERARCHICAL COMMUNITY DETECTION - RECURSIVE BISECTION")
print("="*70)

initial_nodes = list(G.nodes())
hierarchy = spectral_bisection_recursive(initial_nodes, G, pos, depth=0, parent_id="")